# Solution D2 · XGBoost importances: gain, weight, cover, permutation and contributions

**Type**: Interpret

## Objective

On a dataset with a **very informative** feature, a moderately informative one, two correlated ones and two noise ones (one continuous and one binary) compare five XGBoost importance measures: `weight` (how many times used), `gain` (average gain), `cover`, permutation on the test set and the mean of the **per-example contributions** (`pred_contribs`).
Check that the contributions **sum to the raw margin** and explain a single prediction.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def generate(n=4000, seed=0):
    rng = np.random.default_rng(seed)
    strong, medium, z = rng.normal(size=(3, n))
    corr_a, corr_b = z + rng.normal(0, 0.3, n), z + rng.normal(0, 0.3, n)
    y = (2 * strong + 0.7 * medium + 1.2 * z + rng.normal(0, 1, n) > 0).astype(int)
    X = pd.DataFrame({"strong": strong, "medium": medium, "corr_a": corr_a, "corr_b": corr_b, "noise_c": rng.normal(size=n), "noise_b": rng.integers(0, 2, n).astype(float)})
    return X, y


X, y = generate()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
model = xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
booster = model.get_booster()
names = list(X.columns)

## Commented solution

In [ ]:
dm = xgb.DMatrix(X_te)
contribs = booster.predict(dm, pred_contribs=True)                       # (examples, features + bias)
table = pd.DataFrame(index=names, columns=["weight", "gain", "cover", "permutation", "contributions"], dtype=float)
for kind in ["weight", "gain", "cover"]:
    table[kind] = pd.Series(booster.get_score(importance_type=kind)).reindex(names).fillna(0)
table["permutation"] = permutation_importance(model, X_te, y_te, n_repeats=10, random_state=0, scoring="roc_auc").importances_mean
table["contributions"] = np.abs(contribs[:, :-1]).mean(0)
table = table.clip(lower=0)
table = table / table.sum()
table.round(3)

In [ ]:
weight_rewards_noise = bool(table.loc["noise_c", "weight"] >= 0.1 and table.loc["noise_c", "permutation"] < 0.02)
gain_relegates_noise = bool(table.loc["noise_c", "gain"] < 0.5 * table.loc["medium", "gain"])
top_by_method = table.idxmax().to_dict()
print("weight rewards the continuous noise:", weight_rewards_noise, "| gain relegates it:", gain_relegates_noise)
print("top feature by method:", top_by_method)

In [ ]:
margin = booster.predict(dm, output_margin=True)
sum_ok = bool(np.allclose(contribs.sum(1), margin, atol=1e-4))
prob_ok = bool(np.allclose(1 / (1 + np.exp(-margin)), model.predict_proba(X_te)[:, 1], atol=1e-4))
print("the sum of the contributions matches the raw margin:", sum_ok, "| the sigmoid of the margin is predict_proba:", prob_ok)

In [ ]:
c0 = pd.Series(contribs[0], index=names + ["bias"])
explanation = c0.reindex(c0.drop("bias").abs().sort_values(ascending=False).index.tolist() + ["bias"])
decisive_feature = explanation.drop("bias").abs().idxmax()
print(explanation.round(3).to_string())
print(f"margin = sum = {explanation.sum():.3f} -> probability {1 / (1 + np.exp(-explanation.sum())):.3f} | decisive feature: {decisive_feature}")

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and close(table.sum().to_numpy(), np.ones(5)), "complete table with columns normalized to sum 1", "divide each column by its sum")
    check(weight_rewards_noise is True and gain_relegates_noise is True, "the continuous noise weighs over 10% for weight, less than half of `medium` for gain and less than 2% for permutation", "weight counts the cuts: many distinct values, many cuts")
    check(top_by_method["gain"] == "strong" and top_by_method["permutation"] == "strong" and top_by_method["contributions"] == "strong", "gain, permutation and contributions put `strong` first", "idxmax per column")
    check(sum_ok is True and prob_ok is True, "the contributions sum to the raw margin; the sigmoid of the margin reproduces predict_proba", "pred_contribs has a final 'bias' column")
    check(len(explanation) == 7 and explanation.index[-1] == "bias" or "bias" in explanation.index, "the explanation includes six features and the constant term", "one column per feature and one for the bias")
    check(decisive_feature in names and abs(explanation[decisive_feature]) == explanation.drop("bias").abs().max(), "the decisive feature is the one with the largest absolute contribution", "argmax of the absolute contributions")
done()

## What you learned

- **`weight` counts the cuts, not the usefulness:** a continuous pure-noise feature (`noise_c`) is used in 13% of the splits (almost as much as the two correlated features, 17-19%); its `gain` share is 3.6% and that of permutation and contributions practically zero. `cover` (10.9%) does not relegate it: it counts how many examples go through its cuts, not how useful they are. The binary noise feature (`noise_b`) has only one possible threshold and gets 1% of `weight`. Like the impurity importance of a tree (M7 D2), `weight` rewards many distinct values.
- **`gain` is the most sensible default** (and the value of `feature_importances_` of the scikit-learn wrapper), but it is computed on the training set and depends on the structure of the trees: here it assigns 46% to `strong` and the permutation gives it 76%. The permutation on unseen data remains the reference; with the correlated `corr_a` and `corr_b` the importance is split between the two copies (M8 C2).
- **The per-example contributions sum exactly to the margin:** `contribs.sum(1) == margin` and the sigmoid of the margin is `predict_proba`. It is an **exact decomposition** of the prediction: each feature "pushes" the log-odds by a certain value relative to the mean prediction (`bias`). The mean of their absolute values is a global importance consistent with the permutation.
- **A local explanation is a table, and it can contradict the global one:** for the first test example the decisive feature is `medium` (+2.29), countered by `corr_b` (−1.17) and `corr_a` (−0.41); `strong`, the most important on average, weighs only +0.17. Sum with the bias: margin 1.00, probability 0.73. It is the foundation of SHAP (`shap.TreeExplainer` computes values of the same kind with a more general method).

## Common pitfall

Reporting `weight` as the "importance of the variables" in a report: it rewards features with many distinct values and makes you overestimate the continuous noise. For a business question use the permutation on validation data and for the single case the contributions.

## Interview question

<details><summary>How do you explain the prediction of an XGBoost model?</summary>

At the global level with the permutation importance on a validation set (avoiding `weight`, which rewards continuous features), at the local level with the per-example contributions (`pred_contribs` or SHAP TreeExplainer): each feature contributes to the log-odds with a signed value, and the sum of the contributions plus the bias is the raw
margin. With correlated features I group the variables before interpreting.

</details>